# RAG Generation — Retrieval + Synthesis, End to End (Milestone 4)

**This notebook builds no index.** It restores the production index exactly as
`09_rag_retrieval.ipynb` does, then adds the piece M3 designed but never ran:
the **synthesis LLM** that turns retrieved chunks into a farmer-facing answer.

## What it needs on Drive

| Artifact | Purpose |
|---|---|
| `agri_knowledge-*.snapshot` (~3.8 GB) | 723,439 vectors + payloads + HNSW graph |
| `manifest.json` | query-side contract: model, prefixes, dim, tiers, fusion weights |

## The generator

`google/gemma-3-4b-it`, 4-bit NF4. This is not a new decision:

- **M3 §4.1** — synthesis LLM is Gemma, 2–4B student
- **M3 §5.4** — an off-the-shelf 2–3B instruct model is named as *"a legitimate
  fallback if distillation is constrained"*. Distillation is constrained: §7.5's
  paired teacher/student set does not exist yet, so this is the sanctioned path.
- **M1 §Deployment** — "Gemma 4B-it / 12B-it, 4-bit quantization (bitsandbytes),
  3.5 GB / 6.7 GB VRAM"

Staying in the Gemma family matters for a reason beyond consistency: §7.5's plan
distils a 12B teacher into this size class, and teacher and student must share a
tokenizer family. Baseline on a different family and the M4 numbers stop being a
valid control for the M5 student.

## What this notebook adds beyond `09`

1. **Tier gate** — `abstain_out_of_scope` returns a refusal and the LLM is
   **never invoked**. §9.9 requires this; here it is enforced and asserted.
2. **§10-compliant prompt assembly** — system rules, cited chunks, tool slot,
   query, output format.
3. **Retrieval hyperparameter sweep** — `top_k`, intent weighting on/off, and the
   fusion weights that M3 Appendix C records as *"chosen by reasoning about source
   suitability, never swept"*.
4. **Generation hyperparameter sweep** — temperature, `max_new_tokens`, context
   depth, and optionally quantization.
5. **Automated scoring** — numeric grounding (does every dose in the answer appear
   in the supplied context?), language match, length compliance, latency split.
6. **M5 artifacts** — one JSONL row per generation so Milestone 5 is analysis over
   a file, with no reruns.

**Run order:** deps → Qdrant → restore → embedder → filters → retrieval tool →
generator → pipeline → sweeps → health check → export.
Budget ~20 min on a T4 for the full pass; the snapshot upload dominates.

> **Before you start:** Gemma is a gated repo. Accept the licence on the model
> page, create a read token, and add it to Colab secrets as `HF_TOKEN`. Without
> it the download fails with a 401 that looks like a network error.

In [ ]:
# Query-time + generation deps. No chunking/preprocessing libraries needed here.
!pip install -q qdrant-client sentence-transformers transformers sentencepiece accelerate bitsandbytes --break-system-packages

## 0. Run Configuration

Every expensive stage is a flag. Flip them off to iterate on one part without
paying for the rest. Defaults give a complete M4 pass in roughly 20 minutes on a
T4 once the snapshot is restored.

In [ ]:
# ---- what to run -----------------------------------------------------------
RUN_EXAMPLES      = True    # worked examples, ~1 min
RUN_RETRIEVAL_SWEEP = True  # retrieval hyperparameters, no LLM, ~2 min
RUN_GEN_SWEEP     = True    # generation hyperparameters, ~6 min on T4
RUN_QUANT_SWEEP   = True   # reloads the model 3x (~15 min) — opt-in
RUN_HEALTH        = True
EXPORT_ARTIFACTS  = True

# ---- generator -------------------------------------------------------------
# Alternatives if the gated download is a problem (note the family break in the
# report if you switch): "Qwen/Qwen2.5-3B-Instruct" is ungated with solid Hindi.
GEN_MODEL_ID   = "google/gemma-3-4b-it"
GEN_LOAD_4BIT  = True

# ---- generation defaults (the baseline every sweep varies one axis from) ----
GEN_BASELINE = dict(
    temperature    = 0.3,
    top_p          = 0.9,
    max_new_tokens = 100,   # M3 §10 caps answers at 60-100 tokens
    ctx_top_k      = 5,     # how many retrieved chunks enter the prompt
)

# ---- where M5 artifacts go -------------------------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/rag_generation_m4"

import json, os, re, time, math
print("config set")

config set


## 1. Qdrant Server

Identical to `09` §1. The index was built with HNSW, which only exists in
**server mode** — `QdrantClient(path=...)` ignores the graph, brute-forces every
query (3.7 s at 107k vs 30–330 ms at 723k), and cannot load a snapshot at all.

Colab has no Docker daemon, so we run the static binary. The **musl** build is
required: the `linux-gnu` asset needs GLIBC 2.38 and Colab ships 2.35.

In [ ]:
import os, subprocess, tarfile, time, requests

QDRANT_STORAGE = "/content/qdrant_serve_storage"
QDRANT_URL     = "http://localhost:6333"
QDRANT_BIN     = "/content/qdrant"
os.makedirs(QDRANT_STORAGE, exist_ok=True)

def qdrant_alive(url=QDRANT_URL, timeout=1):
    try:
        return requests.get(f"{url}/readyz", timeout=timeout).ok
    except Exception:
        return False

def binary_ok(path=QDRANT_BIN):
    """Existence is not enough — a glibc-linked build downloads fine and only
    fails at exec, so actually run it."""
    if not os.path.exists(path):
        return False
    try:
        return subprocess.run([path, "--version"], capture_output=True,
                              timeout=60).returncode == 0
    except Exception:
        return False

if qdrant_alive():
    print("Qdrant already running on :6333")
else:
    if not binary_ok():
        if os.path.exists(QDRANT_BIN):
            os.remove(QDRANT_BIN)
        rel = requests.get("https://api.github.com/repos/qdrant/qdrant/releases/latest",
                           timeout=30).json()
        asset = None
        for suffix in ("x86_64-unknown-linux-musl.tar.gz",     # static — no libc dependency
                       "x86_64-unknown-linux-gnu.tar.gz"):     # needs GLIBC 2.38; last resort
            asset = next((a for a in rel["assets"] if a["name"].endswith(suffix)), None)
            if asset:
                break
        if asset is None:
            raise RuntimeError("no linux x86_64 asset in " + rel["tag_name"])
        print(f"downloading qdrant {rel['tag_name']} ({asset['name']}) ...")
        with open("/content/q.tar.gz", "wb") as f:
            f.write(requests.get(asset["browser_download_url"], timeout=600).content)
        with tarfile.open("/content/q.tar.gz") as t:
            try:
                t.extractall("/content", filter="data")
            except TypeError:
                t.extractall("/content")
        os.chmod(QDRANT_BIN, 0o755)
        if not binary_ok():
            raise RuntimeError("downloaded qdrant will not execute here")

    env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=QDRANT_STORAGE,
                           QDRANT__TELEMETRY_DISABLED="true")
    qdrant_proc = subprocess.Popen([QDRANT_BIN], env=env, cwd="/content",
                                   stdout=open("/content/qdrant.log", "w"),
                                   stderr=subprocess.STDOUT)
    for _ in range(90):
        if qdrant_alive():
            break
        if qdrant_proc.poll() is not None:
            raise RuntimeError("qdrant exited early — /content/qdrant.log:\n"
                               + open("/content/qdrant.log").read()[-1500:])
        time.sleep(1)
    else:
        raise RuntimeError("qdrant not ready in 90s — see /content/qdrant.log")
    print("Qdrant server ready on :6333")

Qdrant already running on :6333


## 2. Restore the Index

Identical to `09` §2. Uploads the snapshot into the running server and reads the
manifest. If the collection is already present the restore is skipped.

In [ ]:
import json, os, time, requests

# --- point this at the folder written by section 8 of the build notebook -----
ARTIFACT_DIR = "/content/drive/MyDrive/rag_production_bge_m3"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"(not on Colab / Drive already available: {type(e).__name__})")

assert os.path.isdir(ARTIFACT_DIR), (
    f"artifact folder not found: {ARTIFACT_DIR}\n"
    "Run section 8 (EXPORT) of 08c_rag_vector_db_bge_m3.ipynb first.")

MANIFEST = json.load(open(os.path.join(ARTIFACT_DIR, "manifest.json"), encoding="utf-8"))
print("manifest:")
for k in ("embed_model", "embed_dim", "query_prefix", "doc_prefix", "collection",
          "n_chunks", "tiers", "built_utc"):
    print(f"   {k:<14} {MANIFEST.get(k)}")

COLLECTION_NAME = MANIFEST["collection"]

_existing = requests.get(f"{QDRANT_URL}/collections", timeout=30).json()["result"]["collections"]
if any(c["name"] == COLLECTION_NAME for c in _existing):
    print(f"\ncollection '{COLLECTION_NAME}' already present — skipping restore")
else:
    snap = os.path.join(ARTIFACT_DIR, MANIFEST["snapshot"])
    assert os.path.exists(snap), f"snapshot missing: {snap}"
    gb = os.path.getsize(snap) / 1e9
    print(f"\nuploading snapshot ({gb:.2f} GB) — the slow step, several minutes ...")
    t0 = time.time()
    with open(snap, "rb") as fh:
        r = requests.post(
            f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots/upload?priority=snapshot",
            files={"snapshot": (MANIFEST["snapshot"], fh)}, timeout=7200)
    r.raise_for_status()
    print(f"restored in {(time.time()-t0)/60:.1f} min")

from qdrant_client import QdrantClient
qdrant_client = QdrantClient(url=QDRANT_URL, timeout=600)
info = qdrant_client.get_collection(COLLECTION_NAME)
print(f"\ncollection '{COLLECTION_NAME}'")
print(f"   points        : {info.points_count:,}")
print(f"   indexed vecs  : {info.indexed_vectors_count:,}")
print(f"   status        : {info.status}")

if info.indexed_vectors_count < info.points_count:
    print("\n[WARN] HNSW not fully built yet; queries work but are slower until it is.")
if MANIFEST.get("n_chunks") and info.points_count != MANIFEST["n_chunks"]:
    print(f"\n[WARN] point count {info.points_count:,} != manifest {MANIFEST['n_chunks']:,}")

Mounted at /content/drive
manifest:
   embed_model    BAAI/bge-m3
   embed_dim      1024
   query_prefix   
   doc_prefix     
   collection     agri_knowledge
   n_chunks       723439
   tiers          {'fallback': 0.56, 'grounded': 0.66}
   built_utc      2026-07-27T13:10:54Z

uploading snapshot (3.80 GB) — the slow step, several minutes ...
restored in 3.2 min

collection 'agri_knowledge'
   points        : 723,439
   indexed vecs  : 723,439
   status        : green


## 3. Embedder and Query Contract

Identical to `09` §3 — every setting read from the manifest, never retyped. The
three that corrupt results silently if they drift: **model**, **prefixes**
(bge-m3 uses none), **max_seq_length**.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = MANIFEST["embed_model"]
MODEL_MAX_TOKENS = MANIFEST["max_seq_length"]
QUERY_PREFIX     = MANIFEST["query_prefix"]
DOC_PREFIX       = MANIFEST["doc_prefix"]

embed_doc   = lambda t: DOC_PREFIX + t
embed_query = lambda t: QUERY_PREFIX + t

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"loading {EMBED_MODEL_NAME} on {device} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
embed_model.max_seq_length = MODEL_MAX_TOKENS

EMBED_DIM = embed_model.get_sentence_embedding_dimension()
assert EMBED_DIM == MANIFEST["embed_dim"], (
    f"dim mismatch: model gives {EMBED_DIM}, index was built with "
    f"{MANIFEST['embed_dim']} — wrong model for this snapshot")
print(f"   dim {EMBED_DIM}  max_seq {MODEL_MAX_TOKENS}  "
      f"prefixes query={QUERY_PREFIX!r} doc={DOC_PREFIX!r}")

TOP_K_DEFAULT  = MANIFEST["top_k_default"]
TIER_GROUNDED  = MANIFEST["tiers"]["grounded"]     # >= : cite-and-answer
TIER_FALLBACK  = MANIFEST["tiers"]["fallback"]     # >= : answer + "verify with KVK"
FUSION_WEIGHTS = MANIFEST["fusion_weights"]        # <  : abstain / out-of-scope

print(f"   tiers: abstain < {TIER_FALLBACK} <= fallback < {TIER_GROUNDED} <= grounded")
print(f"   fusion: {FUSION_WEIGHTS}")

if device == "cpu":
    print("\n[WARN] No GPU. Retrieval works; 4-bit generation needs CUDA and will")
    print("       be skipped. Switch to a T4 runtime for the generation sections.")

loading BAAI/bge-m3 on cuda ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

   dim 1024  max_seq 512  prefixes query='' doc=''
   tiers: abstain < 0.56 <= fallback < 0.66 <= grounded
   fusion: {'policy': {'pdf': 2.0, 'kcc': 0.5}, 'field_practice': {'pdf': 0.5, 'kcc': 2.0}, 'general': {'pdf': 1.0, 'kcc': 1.0}}


/tmp/ipykernel_630/1560888525.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embed_model.get_sentence_embedding_dimension()


## 4. Filter Canonicalization

Identical to `09` §4. `crop` and `district` filter values must normalise exactly
as they did at ingestion or a filter silently matches nothing — the bug that made
`crop="rice"` match **zero** of 112,269 rice chunks with no error.

In [ ]:
import re

# District renames/typos -> canonical post-bifurcation names.
DISTRICT_CANON = {
    "allahabad": "prayagraj", "faizabad": "ayodhya",
    "prabuddh nagar": "shamli", "prabudh nagar": "shamli",
    "bhim nagar": "sambhal", "panchsheel nagar": "hapur",
    "jyotiba phule nagar": "amroha", "jyotibaphule nagar": "amroha",
    "kanshi ram nagar": "kasganj", "kanshiram nagar": "kasganj",
    "chhatrapati shahuji maharaj nagar": "amethi",
    "mahamaya nagar": "hathras", "ramabai nagar": "kanpur dehat",
    "banaras": "varanasi", "kashi": "varanasi",
    "kanpur city": "kanpur nagar", "maharahganj": "maharajganj",
    "sant ravidas nagar": "bhadohi",
}

def canon_district(raw):
    """Lowercase, collapse spaces, apply the shared rename/typo map."""
    if not raw or str(raw).lower() in ("unknown", "nan", "none", ""):
        return None
    d = re.sub(r"\s+", " ", str(raw).strip().lower())
    return DISTRICT_CANON.get(d, d)

CROP_CANON = {
    "rice": "rice", "paddy": "rice", "dhan": "rice", "chawal": "rice",
    "wheat": "wheat", "gehun": "wheat", "gehu": "wheat", "kanak": "wheat",
    "maize": "maize", "makka": "maize", "makai": "maize", "bhutta": "maize", "corn": "maize",
    "sugarcane": "sugarcane", "ganna": "sugarcane", "noble cane": "sugarcane",
    "mustard": "mustard", "sarson": "mustard", "raya": "mustard",
    "indian mustard": "mustard", "indian rapeseed and mustard": "mustard", "yellow sarson": "mustard",
    "urad": "urad", "black gram": "urad", "urd": "urad", "urd bean": "urad",
    "gram": "gram", "bengal gram": "gram", "chana": "gram", "chick pea": "gram", "kabuli": "gram",
    "moong": "moong", "green gram": "moong", "moong bean": "moong", "mung": "moong",
    "arhar": "arhar", "pigeon pea": "arhar", "red gram": "arhar", "tur": "arhar",
    "masur": "masur", "lentil": "masur",
    "okra": "okra", "bhindi": "okra", "ladysfinger": "okra",
    "bajra": "bajra", "pearl millet": "bajra", "bulrush millet": "bajra", "spiked millet": "bajra",
    "jowar": "jowar", "sorghum": "jowar", "great millet": "jowar",
    "barley": "barley", "jau": "barley",
    "sesame": "sesame", "til": "sesame", "gingelly": "sesame", "sesamum": "sesame",
    "groundnut": "groundnut", "pea nut": "groundnut", "peanut": "groundnut", "mung phalli": "groundnut",
    "colocasia": "arvi", "arvi": "arvi", "arbi": "arvi", "arum": "arvi",
    "cotton": "cotton", "kapas": "cotton",
    "soybean": "soybean", "bhat": "soybean",
    "linseed": "linseed", "alsi": "linseed",
    "spinach": "spinach", "palak": "spinach",
    "methi": "fenugreek", "fenugreek": "fenugreek",
    "rajma": "rajma", "french bean": "rajma",
    "sunflower": "sunflower", "suryamukhi": "sunflower",
    "finger millet": "ragi", "fingermillet": "ragi", "ragi": "ragi", "mandika": "ragi",
    "pea": "pea", "peas": "pea", "matar": "pea", "field peas": "pea", "garden peas": "pea",
}

_CROP_PAREN = re.compile(r"^([^(]+?)\s*\((.*)\)\s*$")
_CROP_NULLS = ("unknown", "nan", "none", "", "na", "n/a", "other", "others")

def canon_crop(raw):
    """Normalise ANY crop surface form to one canonical token.

    Order: whole string -> base before '(' -> each alias inside '()' -> the base.
    Applied identically to payload values at ingestion and to the caller's filter
    term at query time, so both sides always land on the same token.
    """
    if raw is None:
        return None
    c = re.sub(r"\s+", " ", str(raw).strip().lower())
    if c in _CROP_NULLS:
        return None
    if c in CROP_CANON:                          # 'rice', 'dhan', 'paddy'
        return CROP_CANON[c]
    m = _CROP_PAREN.match(c)
    if m:
        base, inner = m.group(1).strip(), m.group(2)
        if base in CROP_CANON:                   # 'paddy (dhan)' -> 'paddy' -> 'rice'
            return CROP_CANON[base]
        for alias in re.split(r"[/,]", inner):   # 'bhindi(okra/ladysfinger)'
            alias = alias.strip()
            if alias in CROP_CANON:
                return CROP_CANON[alias]
        return base                              # unmapped: keep the English base
    return c

print("canonicalizers ready")

canonicalizers ready


## 5. The Retrieval Tool

`search_agri_knowledge(...)` is reproduced from `09` §5 **unchanged in signature
and behaviour** — it is the frozen contract the agent layer calls.

One structural change: the body moves into `_search_core(...)`, which exposes the
knobs the §8 sweep needs (fusion weight override, per-stage timing). The public
function is a thin wrapper that pins those to manifest values, so the production
path cannot drift while the sweep varies them.

**Tiers are decided on the RAW cosine, never the fused score** — fusion multiplies
by up to 2.0, which would otherwise manufacture confidence out of a weighting choice.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

RAG_TOOL_SPEC = {
    "name": "search_agri_knowledge",
    "description": (
        "Search the UP agricultural knowledge base: government scheme guidelines, "
        "pest/disease advisories and district contingency plans (PDF corpus) plus "
        "Kisan Call Centre farmer Q&A with expert answers (KCC corpus). Returns "
        "cited chunks with a relevance tier. Query may be English or Hindi."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query":       {"type": "string", "description": "The farmer's question, English or Hindi"},
            "top_k":       {"type": "integer", "default": 5, "minimum": 1, "maximum": 20},
            "intent":      {"type": "string", "enum": ["policy", "field_practice", "general"],
                            "description": "Weights the PDF-vs-KCC fusion; default general"},
            "source_type": {"type": "string", "enum": ["pdf", "kcc"],
                            "description": "Pin one corpus; omit for weighted search over both"},
            "doc_category": {"type": "string", "enum": ["scheme_eligibility", "crop_advisory",
                                                        "contingency_plan", "policy_guideline"],
                             "description": "PDF corpus only"},
            "query_type":  {"type": "string", "description": "KCC only, e.g. 'Plant Protection'"},
            "crop":        {"type": "string", "description": "Canonical crop name (rice, wheat, ...)"},
            "district":    {"type": "string", "description": "Canonical UP district name"},
            "season":      {"type": "string", "enum": ["Rabi", "Kharif", "Zaid"], "description": "KCC only"},
            "language":    {"type": "string", "enum": ["en", "hi", "mixed"]},
            "year_from":   {"type": "integer", "description": "Only content from this year onward"},
            "only_tables": {"type": "boolean", "description": "PDF dosage/scheme tables only"},
        },
        "required": ["query"],
    },
}


def _citation(p):
    if p.get("source_type") == "pdf":
        return {"corpus": "pdf", "file": p.get("filename"),
                "pages": [p.get("page_start"), p.get("page_end")],
                "section": p.get("heading_hierarchy") or None,
                "doc_category": p.get("doc_category"),
                "district": p.get("district"), "year": p.get("year")}
    return {"corpus": "kcc", "record": "KCC Q&A", "crop": p.get("crop"),
            "district": p.get("district"), "season": p.get("season"),
            "query_type": p.get("query_type"), "year": p.get("year")}


def _search_core(query, top_k=None, intent="general", source_type=None,
                 doc_category=None, query_type=None, crop=None, district=None,
                 season=None, language=None, year_from=None, only_tables=None,
                 fusion_override=None):
    """Instrumented core. `fusion_override` and the timing block exist for the
    §8 sweep; the public tool below never uses them."""
    top_k   = TOP_K_DEFAULT if top_k is None else top_k
    wtable  = fusion_override if fusion_override is not None else FUSION_WEIGHTS
    weights = wtable.get(intent, wtable.get("general", {}))
    timing  = {"embed_ms": 0.0, "search_ms": 0.0}

    def sub_search(stype, qvec):
        must = [FieldCondition(key="source_type", match=MatchValue(value=stype))]
        if doc_category: must.append(FieldCondition(key="doc_category", match=MatchValue(value=doc_category)))
        if query_type:   must.append(FieldCondition(key="query_type", match=MatchValue(value=query_type)))
        if crop:         must.append(FieldCondition(key="crop", match=MatchValue(value=canon_crop(crop))))
        if district:     must.append(FieldCondition(key="district", match=MatchValue(value=canon_district(district))))
        if season:       must.append(FieldCondition(key="season", match=MatchValue(value=season)))
        if language:     must.append(FieldCondition(key="language", match=MatchValue(value=language)))
        if year_from:    must.append(FieldCondition(key="year", range=Range(gte=year_from)))
        if only_tables:  must.append(FieldCondition(key="has_table", match=MatchValue(value=True)))
        return qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            query_filter=Filter(must=must),
            limit=top_k,
            with_payload=True,
        ).points

    try:
        t0 = time.time()
        # bge-m3 takes no prefixes; embed_query applies whatever the manifest says.
        qvec = embed_model.encode(embed_query(query), normalize_embeddings=True).tolist()
        timing["embed_ms"] = (time.time() - t0) * 1000

        t0 = time.time()
        sources = [source_type] if source_type else ["pdf", "kcc"]
        hits = []
        for stype in sources:
            for h in sub_search(stype, qvec):
                hits.append({
                    "raw_score": round(float(h.score), 4),
                    "fused_score": round(float(h.score) * weights.get(stype, 1.0), 4),
                    "text": h.payload.get("text", ""),
                    "source_type": stype,
                    "has_table": bool(h.payload.get("has_table", False)),
                    "chunk_id": h.payload.get("chunk_id"),
                    "citation": _citation(h.payload),
                })
        hits.sort(key=lambda x: x["fused_score"], reverse=True)
        hits = hits[:top_k]
        timing["search_ms"] = (time.time() - t0) * 1000
    except Exception as e:
        return {"query": query, "tier": "error", "top_score": 0.0,
                "results": [], "error": str(e), "timing": timing}

    best_raw = max((h["raw_score"] for h in hits), default=0.0)  # tier on RAW cosine
    tier = ("grounded" if best_raw >= TIER_GROUNDED
            else "fallback_with_disclaimer" if best_raw >= TIER_FALLBACK
            else "abstain_out_of_scope")
    return {"query": query, "intent": intent, "tier": tier,
            "top_score": round(best_raw, 4), "results": hits, "timing": timing}


def search_agri_knowledge(query, top_k=TOP_K_DEFAULT, intent="general", source_type=None,
                          doc_category=None, query_type=None, crop=None, district=None,
                          season=None, language=None, year_from=None, only_tables=None):
    """LLM tool entrypoint: JSON-in/JSON-out, never raises, always cites.
    Frozen contract — identical behaviour to 09_rag_retrieval.ipynb."""
    return _search_core(query, top_k=top_k, intent=intent, source_type=source_type,
                        doc_category=doc_category, query_type=query_type, crop=crop,
                        district=district, season=season, language=language,
                        year_from=year_from, only_tables=only_tables)


print("Tool ready: search_agri_knowledge(...)")
_s = search_agri_knowledge("interest subvention on crop loans", top_k=3, intent="policy")
print(f"smoke: tier={_s['tier']} top={_s['top_score']} "
      f"sources={[r['source_type'] for r in _s['results']]}")
assert _s["tier"] != "error", _s.get("error")

Tool ready: search_agri_knowledge(...)
smoke: tier=grounded top=0.7546 sources=['pdf', 'pdf', 'pdf']


## 6. The Generator

Gemma 4B-it at 4-bit NF4. Three details that are easy to get wrong:

- **Compute dtype.** A T4 is Turing and has no bf16. Detected below and set to
  fp16 accordingly; hardcoding bf16 fails at load on the most common free runtime.
- **No system role.** Gemma's chat template has no `system` turn — a system
  message raises. §10's system instructions are folded into the first user turn.
- **Model class.** Gemma 3's 4B checkpoint is multimodal, so `AutoModelForCausalLM`
  may not resolve it. The loader falls through to the image-text-to-text class and
  generates text-only either way.

The generator config is recorded alongside the retrieval config so §11's export
carries the full contract, per §9.11's manifest discipline.

In [ ]:
import torch
GEN_AVAILABLE = torch.cuda.is_available()

# HF token — Gemma is a gated repo. A missing token surfaces as a 401 that reads
# like a network failure, so check it explicitly rather than letting it fail deep.
if "HF_TOKEN" not in os.environ:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            print("HF_TOKEN loaded from Colab secrets")
    except Exception:
        pass
if "HF_TOKEN" not in os.environ:
    print("[WARN] No HF_TOKEN set. Gated repos (Gemma) will fail with 401.")
    print("       Accept the licence on the model page, then add HF_TOKEN to")
    print("       Colab secrets (key icon, left sidebar) and re-run this cell.")

gen_model = gen_tok = None
GEN_LOAD_INFO = {}

def load_generator(model_id=GEN_MODEL_ID, load_4bit=GEN_LOAD_4BIT):
    """Returns (tokenizer, model, info). Safe to call repeatedly."""
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    kw = dict(device_map="auto", token=os.environ.get("HF_TOKEN"))
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)
    else:
        kw["torch_dtype"] = compute_dtype

    tok = AutoTokenizer.from_pretrained(model_id, token=os.environ.get("HF_TOKEN"))
    t0 = time.time()
    try:
        mdl = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    except Exception:
        # Gemma 3 4B/12B are multimodal checkpoints; text generation still works.
        from transformers import AutoModelForImageTextToText
        mdl = AutoModelForImageTextToText.from_pretrained(model_id, **kw)
    mdl.eval()
    load_s = time.time() - t0

    vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    info = {"model_id": model_id, "quantization": "nf4-4bit" if load_4bit else str(compute_dtype),
            "compute_dtype": str(compute_dtype), "load_sec": round(load_s, 1),
            "vram_gb": round(vram, 2)}
    print(f"loaded {model_id}  [{info['quantization']}]  "
          f"{load_s:.0f}s  {vram:.2f} GB VRAM")
    return tok, mdl, info

if GEN_AVAILABLE:
    gen_tok, gen_model, GEN_LOAD_INFO = load_generator()
else:
    print("[SKIP] No CUDA — generation sections will be skipped, retrieval still runs.")

HF_TOKEN loaded from Colab secrets


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

loaded google/gemma-3-4b-it  [nf4-4bit]  1183s  5.51 GB VRAM


## 7. Prompt Assembly and the Tier Gate

### The prompt contract (M3 §10)

```
[System Instructions — role, safety rules, output format]
[Retrieved Chunks — with citation IDs and relevance tier]
[Structured Tool Data — yield estimate, price, profitability range]
[Farmer Query]
[Output Format Instructions]
```

The tool-data block is present but empty: the GBT yield model and the mandi/weather
adapters are separate M4 workstreams. Keeping the slot in the prompt now means the
prompt shape does not change when they land.

### The gate

**`abstain_out_of_scope` must never reach the generator.** §9.9 makes this a
safety requirement, not an optimization — the system emits pesticide dosages, and
a fluent, well-cited answer built on a 0.45-cosine chunk is more dangerous than no
answer. The check is asserted in §10's health suite, not merely intended.

The `fallback_with_disclaimer` tier does generate, with the KVK verification line
appended by the pipeline rather than requested from the model — a rule the model
cannot forget.

In [ ]:
SYSTEM_RULES = (
    "You are an agricultural advisory assistant for farmers in Uttar Pradesh, India.\n"
    "RULES:\n"
    "1. Answer ONLY from the numbered CONTEXT below. Never use outside knowledge.\n"
    "2. Never invent a dosage, price, date or scheme name. If the context does not "
    "contain a specific figure, say the figure is not available.\n"
    "3. The CONTEXT below is mostly in Hindi. Do NOT copy its language. "
    "Your reply language is stated at the very end of this message.\n"
    "4. Cite the context numbers you used, like [1] or [2].\n"
    "5. Be concise and practical: 60-100 tokens, no preamble, no repetition.\n"
)

OUTPUT_FORMAT = (
    "FORMAT: one short paragraph of practical advice, then a final line "
    "'Sources: [n]' listing the context numbers used."
)
LANG_NAME = {
    "hi": "Hindi (Devanagari script)",
    "en": "English",
    "hinglish": "Hinglish (Hindi words written in Latin/Roman script)",
}

KVK_DISCLAIMER = {
    "en": "Please verify with your local KVK before applying.",
    "hi": "कृपया उपयोग से पहले अपने नजदीकी केवीके से पुष्टि कर लें।",
}

ABSTAIN_MSG = {
    "en": ("This question is outside the agricultural knowledge base I can answer "
           "from. Please ask about crops, pests, fertilisers, or government "
           "agriculture schemes in Uttar Pradesh."),
    "hi": ("यह प्रश्न मेरे कृषि ज्ञान आधार से बाहर है। कृपया फसल, कीट, उर्वरक या सरकारी कृषि योजनाओं के बारे में पूछें।"),
}


def deva_ratio(s):
    """Share of alphabetic characters that are Devanagari. Used both to pick the
    reply language for canned messages and to score language match in §9."""
    letters = [ch for ch in s if ch.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for ch in letters if "ऀ" <= ch <= "ॿ") / len(letters)


def detect_lang(text):
    r = deva_ratio(text)
    return "hi" if r > 0.4 else "en"


def format_context(hits, ctx_top_k=5, char_cap=700):
    """Numbered, citable context block. Chunks are capped so a long KCC answer
    cannot crowd the prompt; the cap is a swept hyperparameter in §8."""
    lines, used = [], []
    for i, h in enumerate(hits[:ctx_top_k], 1):
        c = h["citation"]
        if c["corpus"] == "pdf":
            pages = c.get("pages") or [None, None]
            where = f"PDF {c.get('file')} p.{pages[0]}"
        else:
            where = (f"KCC | crop={c.get('crop')} | district={c.get('district')} "
                     f"| {c.get('year')}")
        txt = re.sub(r"\s+", " ", h["text"]).strip()[:char_cap]
        lines.append(f"[{i}] ({where}) score={h['raw_score']:.3f}\n{txt}")
        used.append({"n": i, "chunk_id": h["chunk_id"], "source_type": h["source_type"],
                     "raw_score": h["raw_score"], "citation": c})
    return "\n\n".join(lines), used


def build_prompt(query, hits, tier, ctx_top_k=5, tool_data=None, char_cap=700):
    ctx, used = format_context(hits, ctx_top_k=ctx_top_k, char_cap=char_cap)
    tool_block = json.dumps(tool_data, ensure_ascii=False) if tool_data else "(none available)"
    lang = detect_lang(query)
    body = (
        f"{SYSTEM_RULES}\n"
        f"RELEVANCE TIER: {tier}\n\n"
        f"CONTEXT:\n{ctx}\n\n"
        f"STRUCTURED TOOL DATA:\n{tool_block}\n\n"
        f"FARMER'S QUESTION:\n{query}\n\n"
        f"{OUTPUT_FORMAT}\n\n"
        f"### WRITE YOUR ENTIRE ANSWER IN {LANG_NAME[lang].upper()}. ###\n"
        f"Even though the context above is in Hindi, answer in "
        f"{LANG_NAME[lang]}."
    )
    try:
        prompt = gen_tok.apply_chat_template(
            [{"role": "user", "content": body}], tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = f"<start_of_turn>user\n{body}<end_of_turn>\n<start_of_turn>model\n"
    return prompt, used


@torch.no_grad()
def generate(prompt, temperature=0.3, top_p=0.9, max_new_tokens=100):
    enc = gen_tok(prompt, return_tensors="pt").to(gen_model.device)
    t0 = time.time()
    out = gen_model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=top_p if temperature > 0 else None,
        pad_token_id=gen_tok.pad_token_id or gen_tok.eos_token_id,
    )
    ms = (time.time() - t0) * 1000
    new_ids = out[0][enc["input_ids"].shape[1]:]
    text = gen_tok.decode(new_ids, skip_special_tokens=True).strip()
    return text, {"gen_ms": ms, "in_tokens": int(enc["input_ids"].shape[1]),
                  "out_tokens": int(new_ids.shape[0]),
                  "tok_per_s": round(len(new_ids) / (ms / 1000), 1) if ms else 0.0}


def answer(query, intent="general", ctx_top_k=None, temperature=None, top_p=None,
           max_new_tokens=None, char_cap=700, retrieval_top_k=None,
           fusion_override=None, **filters):
    """End-to-end: retrieve -> tier gate -> (generate | refuse) -> package.

    Returns one dict, which is also exactly the row written to the M5 JSONL."""
    cfg = dict(GEN_BASELINE)
    if ctx_top_k      is not None: cfg["ctx_top_k"]      = ctx_top_k
    if temperature    is not None: cfg["temperature"]    = temperature
    if top_p          is not None: cfg["top_p"]          = top_p
    if max_new_tokens is not None: cfg["max_new_tokens"] = max_new_tokens

    rk = retrieval_top_k if retrieval_top_k is not None else max(cfg["ctx_top_k"], TOP_K_DEFAULT)
    r = _search_core(query, top_k=rk, intent=intent, fusion_override=fusion_override, **filters)
    lang = detect_lang(query)

    row = {"query": query, "lang": lang, "intent": intent, "tier": r["tier"],
           "top_score": r["top_score"], "config": cfg,
           "timing": dict(r.get("timing", {})), "generated": False,
           "answer": None, "context_used": [], "gen_stats": {},
           "retrieved": [{"chunk_id": h["chunk_id"], "source_type": h["source_type"],
                          "raw_score": h["raw_score"], "citation": h["citation"]}
                         for h in r.get("results", [])]}

    if r["tier"] == "error":
        row["answer"] = None
        row["error"] = r.get("error")
        return row

    # --- the gate: below the abstain threshold the LLM is never invoked --------
    if r["tier"] == "abstain_out_of_scope":
        row["answer"] = ABSTAIN_MSG[lang]
        row["timing"]["gen_ms"] = 0.0
        return row

    if not GEN_AVAILABLE or gen_model is None:
        row["answer"] = None
        row["error"] = "generator unavailable (no CUDA / model not loaded)"
        return row

    prompt, used = build_prompt(query, r["results"], r["tier"],
                                ctx_top_k=cfg["ctx_top_k"], char_cap=char_cap)
    text, stats = generate(prompt, temperature=cfg["temperature"],
                           top_p=cfg["top_p"], max_new_tokens=cfg["max_new_tokens"])

    # Disclaimer is appended by rule, not requested from the model — a rule the
    # model cannot forget or paraphrase away.
    if r["tier"] == "fallback_with_disclaimer":
        text = f"{text}\n\n{KVK_DISCLAIMER[lang]}"

    row.update(generated=True, answer=text, context_used=used,
               gen_stats=stats, prompt_chars=len(prompt))
    row["timing"]["gen_ms"] = stats["gen_ms"]
    row["timing"]["total_ms"] = (row["timing"].get("embed_ms", 0)
                                 + row["timing"].get("search_ms", 0) + stats["gen_ms"])
    return row


print("pipeline ready: answer(query, intent=..., **filters)")

pipeline ready: answer(query, intent=..., **filters)


## 8. Worked Examples

The same query spread `09` uses, now carried through to a generated answer. Read
these rather than trusting the scores — the point of the exercise is to see
whether a plausible-looking retrieval produces a defensible answer.

Watch the fourth one especially. `gehu me pila ratua` is M3 §14's documented
failure: retrieval returns potato leaf-yellowing for a wheat-rust question. If it
scores above the abstain threshold, the generator will wrap fluent, cited prose
around the wrong chunk — the §16.2 silent-failure pattern with better packaging.

In [ ]:
def show(row, width=96):
    print("=" * width)
    print(f"Q     : {row['query']}")
    print(f"tier  : {row['tier']}   top={row['top_score']:.3f}   "
          f"lang={row['lang']}   generated={row['generated']}")
    t = row.get("timing", {})
    if t:
        print(f"timing: embed={t.get('embed_ms',0):.0f}ms  search={t.get('search_ms',0):.0f}ms  "
              f"gen={t.get('gen_ms',0):.0f}ms  total={t.get('total_ms',0):.0f}ms")
    if row.get("error"):
        print(f"ERROR : {row['error'][:200]}")
    print(f"\nANSWER:\n{row['answer']}")
    if row["context_used"]:
        print("\nCONTEXT USED:")
        for c in row["context_used"]:
            cit = c["citation"]
            where = (f"{cit.get('file')} p.{(cit.get('pages') or [None])[0]}"
                     if cit["corpus"] == "pdf"
                     else f"KCC {cit.get('crop')}/{cit.get('district')}/{cit.get('year')}")
            print(f"  [{c['n']}] {c['raw_score']:.3f} <{c['source_type']}> {where}")
    print()

if RUN_EXAMPLES:
    demo = [
        ("who is eligible for interest subvention on crop loans", dict(intent="policy")),
        ("how much urea should be applied in wheat at tillering stage", dict(intent="field_practice")),
        ("टमाटर के पौधे में पत्तियां मुड़ रही हैं क्या करें", {}),
        ("gehu me pila ratua lag gaya hai kya kare", dict(intent="field_practice")),
        ("approved fungicides and dosage for rice blast", dict(only_tables=True)),
        ("how do I repair my motorcycle engine", {}),   # must abstain, must NOT generate
    ]
    EXAMPLE_ROWS = [answer(q, **kw) for q, kw in demo]
    for r in EXAMPLE_ROWS:
        show(r)
else:
    EXAMPLE_ROWS = []
    print("[SKIP] RUN_EXAMPLES = False")

Q     : who is eligible for interest subvention on crop loans
tier  : grounded   top=0.681   lang=en   generated=True
timing: embed=356ms  search=12850ms  gen=13164ms  total=26370ms

ANSWER:
Individual farmers, joint borrowers, tenant farmers, oral lessees, and share croppers are eligible for interest subvention on short-term crop loans under the Kisan Credit Card (KCC) scheme [4]. The interest subvention rate is 1.50% for the 2025-26 financial year [5].  Loans are capped at ₹3 lakh.

Sources: [4], [5]

CONTEXT USED:
  [1] 0.681 <pdf> MODIFIED_INTEREST_SUBVENTION_SCHEME.pdf p.1
  [2] 0.680 <pdf> MODIFIED_INTEREST_SUBVENTION_SCHEME.pdf p.1
  [3] 0.679 <pdf> MODIFIED_INTEREST_SUBVENTION_SCHEME.pdf p.1
  [4] 0.638 <pdf> Kisan_Credit_Card_(KCC)_Scheme.pdf p.2
  [5] 0.624 <pdf> MODIFIED_INTEREST_SUBVENTION_SCHEME.pdf p.1

Q     : how much urea should be applied in wheat at tillering stage
tier  : grounded   top=0.714   lang=en   generated=True
timing: embed=22ms  search=10292ms  gen=7546ms 

## 9. The Evaluation Set

A fixed, annotated query set — the same one every sweep below is scored against,
so configurations are comparable to each other and to whatever M5 runs.

M3 §14 records that the current tier thresholds rest on **14 self-authored
questions** and calls widening that a *safety* task rather than a tuning one. This
set is 24 queries and still self-authored, so it is a step, not a resolution. It
is annotated with what a correct system should do, which the 14 were not:

| Field | Meaning |
|---|---|
| `expect_answer` | `True` = must reach generation; `False` = must abstain |
| `expect_source` | corpus that should dominate (`pdf` / `kcc` / `None`) |
| `expect_crop` | canonical crop that should appear in the hits |
| `group` | policy / field / hindi / hinglish / gap / offdomain |

`gap` rows are M3 §14's known corpus holes (mandi prices, scheme content thin in
KCC). They are expected to score low — recorded so a future PDF-corpus expansion
has a baseline to beat.

In [ ]:
EVAL_SET = [
    # --- policy: should route PDF-heavy -------------------------------------
    dict(q="who is eligible for interest subvention on crop loans",
         intent="policy", group="policy", expect_answer=True, expect_source="pdf", expect_crop=None),
    dict(q="pm kisan samman nidhi eligibility and benefits",
         intent="policy", group="policy", expect_answer=True, expect_source="pdf", expect_crop=None),
    dict(q="what documents are needed for crop insurance claim",
         intent="policy", group="policy", expect_answer=True, expect_source="pdf", expect_crop=None),
    dict(q="soil health card scheme how to apply",
         intent="policy", group="policy", expect_answer=True, expect_source="pdf", expect_crop=None),

    # --- field practice, English: should route KCC-heavy ---------------------
    dict(q="how much urea should be applied in wheat at tillering stage",
         intent="field_practice", group="field", expect_answer=True, expect_source="kcc", expect_crop="wheat"),
    dict(q="fall army worm control in maize",
         intent="field_practice", group="field", expect_answer=True, expect_source="kcc", expect_crop="maize"),
    dict(q="red rot disease treatment in sugarcane",
         intent="field_practice", group="field", expect_answer=True, expect_source="kcc", expect_crop="sugarcane"),
    dict(q="best fertilizer dose for paddy nursery",
         intent="field_practice", group="field", expect_answer=True, expect_source="kcc", expect_crop="rice"),
    dict(q="late blight control in potato crop",
         intent="field_practice", group="field", expect_answer=True, expect_source="kcc", expect_crop="potato"),
    dict(q="aphid attack on mustard what to spray",
         intent="field_practice", group="field", expect_answer=True, expect_source="kcc", expect_crop="mustard"),

    # --- Hindi (Devanagari) --------------------------------------------------
    dict(q="गेहूं में पीला रतुआ की रोकथाम कैसे करें",
         intent="field_practice", group="hindi", expect_answer=True, expect_source="kcc", expect_crop="wheat"),
    dict(q="धान की नर्सरी में पीली पत्ती हो रही है",
         intent="field_practice", group="hindi", expect_answer=True, expect_source="kcc", expect_crop="rice"),
    dict(q="टमाटर के पौधे में पत्तियां मुड़ रही हैं क्या करें",
         intent="field_practice", group="hindi", expect_answer=True, expect_source="kcc", expect_crop=None),
    dict(q="गन्ने में लाल सड़न रोग का इलाज",
         intent="field_practice", group="hindi", expect_answer=True, expect_source="kcc", expect_crop="sugarcane"),

    # --- Hinglish (transliterated) ------------------------------------------
    dict(q="dhan ki nursery me pili patti ho rahi hai kya karein",
         intent="field_practice", group="hinglish", expect_answer=True, expect_source="kcc", expect_crop="rice"),
    dict(q="gehu me pila ratua lag gaya hai kya kare",   # M3 §14 known failure
         intent="field_practice", group="hinglish", expect_answer=True, expect_source="kcc", expect_crop="wheat"),
    dict(q="aloo ki fasal me jhulsa rog ki dawa bataye",
         intent="field_practice", group="hinglish", expect_answer=True, expect_source="kcc", expect_crop="potato"),
    dict(q="ganne me kide lag gaye hain kaun si dawa dale",
         intent="field_practice", group="hinglish", expect_answer=True, expect_source="kcc", expect_crop="sugarcane"),

    # --- known corpus gaps (M3 §14) — recorded, not expected to succeed ------
    dict(q="mandi bhav for wheat today in lucknow",
         intent="general", group="gap", expect_answer=False, expect_source=None, expect_crop=None),
    dict(q="onion market price today uttar pradesh",
         intent="general", group="gap", expect_answer=False, expect_source=None, expect_crop=None),

    # --- off-domain: MUST abstain, MUST NOT generate -------------------------
    dict(q="how do I repair my motorcycle engine",
         intent="general", group="offdomain", expect_answer=False, expect_source=None, expect_crop=None),
    dict(q="best chess opening strategy for beginners",
         intent="general", group="offdomain", expect_answer=False, expect_source=None, expect_crop=None),
    dict(q="शेयर बाजार में निवेश कैसे करें",
         intent="general", group="offdomain", expect_answer=False, expect_source=None, expect_crop=None),
    dict(q="who won the cricket world cup in 2011",
         intent="general", group="offdomain", expect_answer=False, expect_source=None, expect_crop=None),
]

from collections import Counter
print(f"EVAL_SET: {len(EVAL_SET)} queries")
for g, n in Counter(e["group"] for e in EVAL_SET).most_common():
    print(f"   {g:<10} {n}")

EVAL_SET: 24 queries
   field      6
   policy     4
   hindi      4
   hinglish   4
   offdomain  4
   gap        2


## 10. Automated Scoring

Four automatable signals. None replaces human judgement — M5 does that — but each
catches a failure mode that a score alone hides.

**Numeric grounding.** Every number in the answer must appear in the supplied
context. This is the machine-checkable half of §10's *"never invent dosage or
price figures"*, and it is the single most important guardrail here: the system
emits pesticide doses, and a hallucinated `50` where the source says `500` is a
field-level safety failure that reads perfectly fluently.

**Language match.** §10 requires the reply to use the farmer's language. Scored by
Devanagari ratio agreement, which catches the common failure of answering a Hindi
question in English.

**Length compliance** against §10's 60–100 token cap, and **citation presence**.

> **Scoring correction (after run 1).** Citation spans are stripped before
> numbers are extracted. The model writes `[1, 3]` and `[1, 2, 3, 4]`, not only
> `[1]`, and the original guard recognised just the single-number form — so every
> index inside a multi-number citation was counted as an invented figure. Run 1
> reported 27/70 answers containing an unsupported number; almost all of those
> were the model citing correctly. `has_citation` had the same blind spot and
> scored properly-cited answers as uncited. Both are fixed; run 1's G2 and G5
> figures should not be quoted.

In [ ]:
NUM_RE = re.compile(r"\d+(?:\.\d+)?")

# A citation span is a bracket holding one or more numbers: [1], [1, 3],
# [1, 2, 3, 4]. The first version of this scorer only recognised the single
# number form, so every index inside a multi-number citation was counted as an
# invented figure -- 27 of 70 answers were flagged, and almost all of those were
# the model citing its sources correctly. Strip whole citation spans before
# looking for numeric claims.
CITE_RE = re.compile(r"\[\s*\d+(?:\s*,\s*\d+)*\s*\]")

def numeric_grounding(answer_text, context_text):
    """Fraction of numbers in the answer that also occur in the context.
    Returns (score, unsupported_list). No numbers -> 1.0, nothing to invent."""
    if not answer_text:
        return 1.0, []
    ctx_nums = set(NUM_RE.findall(context_text or ""))
    # Citation spans are formatting, not claims -- remove them before extracting.
    ans_nums = NUM_RE.findall(CITE_RE.sub(" ", answer_text))
    if not ans_nums:
        return 1.0, []
    unsupported = [n for n in ans_nums if n not in ctx_nums]
    return 1.0 - len(unsupported) / len(ans_nums), unsupported


def language_match(query, answer_text):
    if not answer_text:
        return None
    qd, ad = deva_ratio(query), deva_ratio(answer_text)
    return 1.0 if (qd > 0.4) == (ad > 0.4) else 0.0


def has_citation(answer_text):
    """True for [1] and for [1, 3] / [1, 2, 3, 4]. The single-number-only
    pattern scored correctly-cited answers as uncited."""
    return bool(answer_text and CITE_RE.search(answer_text))


def score_row(row):
    """Attach scores in place, return the same row (so it can be dumped as-is)."""
    ctx = " ".join(h.get("text", "") for h in row.get("_ctx_texts", []))
    if not ctx and row.get("context_used"):
        ctx = row.get("_ctx_blob", "")
    ng, unsup = numeric_grounding(row.get("answer"), ctx)
    row["scores"] = {
        "numeric_grounding": round(ng, 3),
        "unsupported_numbers": unsup,
        "language_match": language_match(row["query"], row.get("answer")),
        "has_citation": has_citation(row.get("answer")),
        "out_tokens": row.get("gen_stats", {}).get("out_tokens"),
        "length_ok": (60 <= (row.get("gen_stats", {}).get("out_tokens") or 0) <= 110)
                     if row.get("generated") else None,
    }
    return row


def answer_scored(query, **kw):
    """answer() + the context blob needed to score grounding, + scores."""
    row = answer(query, **kw)
    blob = ""
    if row.get("generated"):
        # Rebuild exactly the text the model saw, so grounding is scored against
        # the real prompt content rather than the full retrieval.
        ids = {c["chunk_id"] for c in row["context_used"]}
        r2 = _search_core(query, top_k=max(row["config"]["ctx_top_k"], TOP_K_DEFAULT),
                          intent=row["intent"])
        blob = " ".join(h["text"] for h in r2["results"] if h["chunk_id"] in ids)
    row["_ctx_blob"] = blob
    return score_row(row)

print("scorers ready")

scorers ready


## 11. Retrieval Hyperparameter Sweep

No LLM involved — this is retrieval only, so it runs on CPU and takes ~2 minutes.

M3 Appendix C records the fusion weights as *"chosen by reasoning about source
suitability, **never swept**"*. This sweeps them, and `top_k` alongside.

**Metrics**, all computed on the annotated eval set:

| Metric | Definition |
|---|---|
| `crop_hit@k` | expected crop appears in the hits (rows with `expect_crop`) |
| `route_acc` | expected corpus dominates the top-3 (rows with `expect_source`) |
| `abstain_acc` | off-domain abstains **and** in-domain does not |
| `margin` | min in-domain top score − max off-domain top score |
| `p50_ms` | median retrieval latency |

`margin` is the load-bearing one: it is the headroom the abstain tier lives in.
A config that lifts `crop_hit` while collapsing `margin` is a worse config, because
it buys ranking quality with the ability to refuse.

In [ ]:
import numpy as np

def eval_retrieval(fusion_override=None, top_k=5, use_intent=True, label=""):
    crop_hits, route_hits, abst_ok, in_scores, off_scores, lats = [], [], [], [], [], []
    for e in EVAL_SET:
        intent = e["intent"] if use_intent else "general"
        r = _search_core(e["q"], top_k=top_k, intent=intent, fusion_override=fusion_override)
        if r["tier"] == "error":
            continue
        lats.append(r["timing"]["embed_ms"] + r["timing"]["search_ms"])
        hits = r["results"]

        if e["expect_crop"]:
            crop_hits.append(any(canon_crop(h["citation"].get("crop")) == e["expect_crop"]
                                 for h in hits))
        if e["expect_source"]:
            top3 = hits[:3]
            dom = sum(h["source_type"] == e["expect_source"] for h in top3)
            route_hits.append(dom >= 2)

        abstained = r["tier"] == "abstain_out_of_scope"
        abst_ok.append(abstained == (not e["expect_answer"]))

        # `gap` rows count toward NEITHER side. They are M3 §14's known corpus
        # holes (no mandi price content exists), so they score as low as
        # off-domain queries by design. Counting them as in-domain pulls
        # min(in_domain) down to the off-domain maximum and reports margin=0.000
        # for every config -- which is what the first run did, while health
        # check R3, which excluded them, correctly reported +0.070.
        if e["group"] == "offdomain":
            off_scores.append(r["top_score"])
        elif e["expect_answer"]:
            in_scores.append(r["top_score"])

    in_dom = [s for s in in_scores if s > 0]
    return {
        "config": label,
        "crop_hit": round(float(np.mean(crop_hits)), 3) if crop_hits else None,
        "route_acc": round(float(np.mean(route_hits)), 3) if route_hits else None,
        "abstain_acc": round(float(np.mean(abst_ok)), 3) if abst_ok else None,
        "margin": round(min(in_dom) - max(off_scores), 4) if in_dom and off_scores else None,
        "p50_ms": round(float(np.percentile(lats, 50)), 1) if lats else None,
    }


RETRIEVAL_SWEEP = []
if RUN_RETRIEVAL_SWEEP:
    base_w = FUSION_WEIGHTS

    # axis 1 — retrieval depth
    for k in (3, 5, 10):
        RETRIEVAL_SWEEP.append(eval_retrieval(top_k=k, label=f"top_k={k}"))

    # axis 2 — intent weighting on vs off (does §9.6 earn its complexity?)
    RETRIEVAL_SWEEP.append(eval_retrieval(top_k=5, use_intent=False, label="intent=off"))

    # axis 3 — fusion weights. Manifest default vs flat vs sharper.
    flat    = {k: {"pdf": 1.0, "kcc": 1.0} for k in base_w}
    sharper = {"policy": {"pdf": 3.0, "kcc": 0.3},
               "field_practice": {"pdf": 0.3, "kcc": 3.0},
               "general": base_w.get("general", {"pdf": 1.0, "kcc": 1.0})}
    RETRIEVAL_SWEEP.append(eval_retrieval(top_k=5, fusion_override=flat,    label="fusion=flat 1.0/1.0"))
    RETRIEVAL_SWEEP.append(eval_retrieval(top_k=5, fusion_override=sharper, label="fusion=3.0/0.3"))
    RETRIEVAL_SWEEP.append(eval_retrieval(top_k=5, label="fusion=manifest (baseline)"))

    hdr = f"{'config':<30}{'crop_hit':>10}{'route_acc':>11}{'abstain':>9}{'margin':>9}{'p50_ms':>9}"
    print("=" * len(hdr)); print("RETRIEVAL HYPERPARAMETER SWEEP"); print("=" * len(hdr))
    print(hdr); print("-" * len(hdr))
    for r in RETRIEVAL_SWEEP:
        f = lambda v, w, p="": "—".rjust(w) if v is None else f"{v:{w}.3f}" if isinstance(v, float) else f"{v:>{w}}"
        print(f"{r['config']:<30}{f(r['crop_hit'],10)}{f(r['route_acc'],11)}"
              f"{f(r['abstain_acc'],9)}{f(r['margin'],9)}{f(r['p50_ms'],9)}")
    print("-" * len(hdr))
    print("Read `margin` first: it is the headroom the abstain tier needs. A config")
    print("that raises crop_hit while shrinking margin trades safety for ranking.")
else:
    print("[SKIP] RUN_RETRIEVAL_SWEEP = False")

RETRIEVAL HYPERPARAMETER SWEEP
config                          crop_hit  route_acc  abstain   margin   p50_ms
------------------------------------------------------------------------------
top_k=3                            0.846      1.000    0.958    0.070 3296.800
top_k=5                            0.923      1.000    0.958    0.070   44.700
top_k=10                           1.000      1.000    0.958    0.070   63.000
intent=off                         0.923      0.889    0.958    0.070   54.200
fusion=flat 1.0/1.0                0.923      0.889    0.958    0.070   53.700
fusion=3.0/0.3                     0.923      1.000    0.958    0.070   61.200
fusion=manifest (baseline)         0.923      1.000    0.958    0.070   43.800
------------------------------------------------------------------------------
Read `margin` first: it is the headroom the abstain tier needs. A config
that raises crop_hit while shrinking margin trades safety for ranking.


## 12. Generation Hyperparameter Sweep

One-factor-at-a-time from the §0 baseline. OFAT rather than a grid is deliberate:
a full grid over four axes is 81 configurations × 24 queries, which does not fit
the milestone, and the axes here are close to independent.

Scored on the answerable subset of the eval set — off-domain rows never generate,
so including them would flatter every configuration equally.

In [ ]:
GEN_SWEEP = []
def _stratified_gen_subset(n_per_group=3):
    """Even coverage across answerable groups.

    Taking the first 10 answerable rows gave 4 policy + 6 field -- every one of
    them English. Health check G3 ('Hindi question -> Hindi answer') therefore
    reported '0/0 mismatched': it passed because it tested nothing, which is the
    same defect M3 §9.10 records in the original retrieval health check."""
    out = []
    for g in ("policy", "field", "hindi", "hinglish"):
        out += [e for e in EVAL_SET
                if e["expect_answer"] and e["group"] == g][:n_per_group]
    return out


GEN_EVAL_SUBSET = _stratified_gen_subset(3)
print(f"generation subset: {len(GEN_EVAL_SUBSET)} queries "
      f"({', '.join(sorted({e['group'] for e in GEN_EVAL_SUBSET}))})")

def eval_generation(label, **overrides):
    rows = [answer_scored(e["q"], intent=e["intent"], **overrides) for e in GEN_EVAL_SUBSET]
    gen = [r for r in rows if r.get("generated")]
    if not gen:
        return {"config": label, "n": 0}, rows
    s = lambda key: [r["scores"][key] for r in gen if r["scores"].get(key) is not None]
    return {
        "config": label, "n": len(gen),
        "grounding": round(float(np.mean(s("numeric_grounding"))), 3),
        "lang_match": round(float(np.mean(s("language_match"))), 3),
        "cited": round(float(np.mean([bool(x) for x in s("has_citation")])), 3),
        "out_tok": round(float(np.mean([r["gen_stats"]["out_tokens"] for r in gen])), 1),
        "tok_s": round(float(np.mean([r["gen_stats"]["tok_per_s"] for r in gen])), 1),
        "gen_ms": round(float(np.mean([r["gen_stats"]["gen_ms"] for r in gen])), 0),
    }, rows

GEN_SWEEP_ROWS = []
if RUN_GEN_SWEEP and GEN_AVAILABLE:
    axes = [
        ("baseline",              {}),
        ("temperature=0.0",       dict(temperature=0.0)),
        ("temperature=0.7",       dict(temperature=0.7)),
        ("max_new_tokens=60",     dict(max_new_tokens=60)),
        ("max_new_tokens=150",    dict(max_new_tokens=150)),
        ("ctx_top_k=3",           dict(ctx_top_k=3)),
        ("ctx_top_k=8",           dict(ctx_top_k=8)),
    ]
    for label, kw in axes:
        res, rows = eval_generation(label, **kw)
        GEN_SWEEP.append(res)
        for r in rows:
            r["sweep_config"] = label
        GEN_SWEEP_ROWS.extend(rows)
        print(f"  done: {label}")

    hdr = (f"{'config':<22}{'n':>4}{'ground':>9}{'lang':>7}{'cited':>7}"
           f"{'out_tok':>9}{'tok/s':>8}{'gen_ms':>9}")
    print("\n" + "=" * len(hdr)); print("GENERATION HYPERPARAMETER SWEEP"); print("=" * len(hdr))
    print(hdr); print("-" * len(hdr))
    for r in GEN_SWEEP:
        if not r.get("n"):
            print(f"{r['config']:<22}{0:>4}   (no generations)"); continue
        print(f"{r['config']:<22}{r['n']:>4}{r['grounding']:>9.3f}{r['lang_match']:>7.2f}"
              f"{r['cited']:>7.2f}{r['out_tok']:>9.1f}{r['tok_s']:>8.1f}{r['gen_ms']:>9.0f}")
    print("-" * len(hdr))
    print("`ground` is the safety number: 1.000 means every figure in every answer")
    print("traced back to a supplied chunk. Anything below 1.0 is an invented dose.")
else:
    print("[SKIP] generation sweep (RUN_GEN_SWEEP off or no CUDA)")

generation subset: 12 queries (field, hindi, hinglish, policy)
  done: baseline
  done: temperature=0.0
  done: temperature=0.7
  done: max_new_tokens=60
  done: max_new_tokens=150
  done: ctx_top_k=3
  done: ctx_top_k=8

GENERATION HYPERPARAMETER SWEEP
config                   n   ground   lang  cited  out_tok   tok/s   gen_ms
---------------------------------------------------------------------------
baseline                12    0.963   1.00   0.92     80.9     7.6    10738
temperature=0.0         12    0.965   1.00   0.92     80.8     7.6    10747
temperature=0.7         12    0.979   1.00   0.83     80.5     7.5    10768
max_new_tokens=60       12    0.864   1.00   0.50     58.2     7.1     8310
max_new_tokens=150      12    0.963   1.00   1.00     86.9     7.6    11284
ctx_top_k=3             12    0.958   1.00   1.00     76.4     8.2     9387
ctx_top_k=8             12    0.974   1.00   0.92     87.3     7.1    12373
--------------------------------------------------------------

## 13. Quantization Ablation *(opt-in)*

Reloads the model at 4-bit, 8-bit and fp16 — roughly 15 minutes. This is the
direct test of M1's *"4-bit, 3.5 GB VRAM"* assumption: does the quantization the
deployment plan assumes cost anything measurable in answer quality?

Off by default because of the reload cost. Turn `RUN_QUANT_SWEEP = True` in §0.

In [ ]:
QUANT_SWEEP = []
if RUN_QUANT_SWEEP and GEN_AVAILABLE:
    import gc
    variants = [("nf4-4bit", dict(load_4bit=True)), ("fp16/bf16", dict(load_4bit=False))]
    for label, kw in variants:
        try:
            del gen_model
        except Exception:
            pass
        gc.collect(); torch.cuda.empty_cache()
        try:
            gen_tok, gen_model, GEN_LOAD_INFO = load_generator(**kw)
        except Exception as e:
            print(f"  {label}: load failed ({type(e).__name__}: {str(e)[:120]})")
            continue
        res, _ = eval_generation(f"quant={label}")
        res["vram_gb"] = GEN_LOAD_INFO["vram_gb"]
        res["load_sec"] = GEN_LOAD_INFO["load_sec"]
        QUANT_SWEEP.append(res)
        print(f"  done: {label}")

    if QUANT_SWEEP:
        hdr = f"{'quantization':<20}{'ground':>9}{'lang':>7}{'tok/s':>8}{'VRAM GB':>10}{'load s':>9}"
        print("\n" + "=" * len(hdr)); print("QUANTIZATION ABLATION"); print("=" * len(hdr))
        print(hdr); print("-" * len(hdr))
        for r in QUANT_SWEEP:
            print(f"{r['config']:<20}{r['grounding']:>9.3f}{r['lang_match']:>7.2f}"
                  f"{r['tok_s']:>8.1f}{r['vram_gb']:>10.2f}{r['load_sec']:>9.0f}")
else:
    print("[SKIP] RUN_QUANT_SWEEP = False (reloads the model 3x)")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

loaded google/gemma-3-4b-it  [nf4-4bit]  36s  5.51 GB VRAM
  done: nf4-4bit


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

loaded google/gemma-3-4b-it  [torch.bfloat16]  35s  10.88 GB VRAM
  done: fp16/bf16

QUANTIZATION ABLATION
quantization           ground   lang   tok/s   VRAM GB   load s
---------------------------------------------------------------
quant=nf4-4bit          0.946   1.00     7.5      5.51       36
quant=fp16/bf16         0.803   1.00     8.9     10.88       35


## 14. Health Check

`09` §8's suite, extended with the checks that only exist once generation is in
the loop. M3 §9.10's principle applies here too: **a health check must exercise
the actual task**. The original retrieval gate passed comfortably while the model
could not tell wheat from mango, because it tested cross-domain separation — the
one thing that model could do.

So these are written to be failable:

- `G1` — abstain must **never** invoke the generator (safety, §9.9)
- `G2` — every number in every answer must trace to context (safety, §10)
- `G3` — a Hindi question must get a Hindi answer (§10 rule 3)
- `G4` — the fallback tier must carry the KVK line
- `G5` — an answered query must cite

In [ ]:
HEALTH = []

def hchk(label, ok, detail=""):
    HEALTH.append((label, "PASS" if ok else "FAIL", detail))

if RUN_HEALTH:
    # --- retrieval-side checks carried over from 09 §8 -----------------------
    r = search_agri_knowledge("fertilizer dose for paddy nursery", source_type="kcc", crop="rice")
    hchk("R1 filter kcc+crop=rice (canon 'Paddy (Dhan)')",
         bool(r["results"]) and all(h["citation"]["crop"] == "rice" for h in r["results"]),
         f"{len(r['results'])} hits")

    r = search_agri_knowledge("contingency plan for delayed monsoon", district="allahabad")
    hchk("R2 district canon allahabad->prayagraj",
         bool(r["results"]) and all(h["citation"].get("district") == "prayagraj" for h in r["results"]),
         f"{len(r['results'])} hits")

    # Same in-domain definition as the §11 sweep: every answerable row,
    # Hinglish included, `gap` rows excluded. The first version listed groups by
    # hand and left Hinglish out, so the sweep and the health check were
    # measuring two different things and reported 0.000 vs +0.070.
    in_s  = [search_agri_knowledge(e["q"], top_k=1)["top_score"]
             for e in EVAL_SET if e["expect_answer"]]
    off_s = [search_agri_knowledge(e["q"], top_k=1)["top_score"]
             for e in EVAL_SET if e["group"] == "offdomain"]
    margin = min(in_s) - max(off_s)
    hchk("R3 domain separation margin > 0.02", margin > 0.02,
         f"in-min={min(in_s):.3f} off-max={max(off_s):.3f} margin={margin:+.3f}")

    # --- generation-side checks (the new ones) -------------------------------
    if GEN_AVAILABLE:
        offdomain = [answer_scored(e["q"], intent=e["intent"])
                     for e in EVAL_SET if e["group"] == "offdomain"]
        hchk("G1 abstain never invokes the generator",
             all(not r["generated"] for r in offdomain),
             f"{sum(r['generated'] for r in offdomain)}/{len(offdomain)} generated")

        answered = GEN_SWEEP_ROWS or [answer_scored(e["q"], intent=e["intent"])
                                      for e in GEN_EVAL_SUBSET]
        answered = [r for r in answered if r.get("generated")]

        bad_ground = [r for r in answered if r["scores"]["numeric_grounding"] < 1.0]
        hchk("G2 every number traces to context", not bad_ground,
             f"{len(bad_ground)}/{len(answered)} answers contain an unsupported number")
        if bad_ground:
            for r in bad_ground[:3]:
                print(f"      unsupported {r['scores']['unsupported_numbers']} in: {r['query'][:60]}")

        hi = [r for r in answered if r["lang"] == "hi"]
        hchk("G3 Hindi question -> Hindi answer",
             all(r["scores"]["language_match"] == 1.0 for r in hi) if hi else True,
             f"{sum(1 for r in hi if r['scores']['language_match'] != 1.0)}/{len(hi)} mismatched")

        fb = [r for r in answered if r["tier"] == "fallback_with_disclaimer"]
        hchk("G4 fallback tier carries the KVK line",
             all(KVK_DISCLAIMER[r["lang"]] in (r["answer"] or "") for r in fb) if fb else True,
             f"{len(fb)} fallback-tier answers")

        hchk("G5 answered queries cite their sources",
             sum(r["scores"]["has_citation"] for r in answered) >= 0.7 * max(len(answered), 1),
             f"{sum(r['scores']['has_citation'] for r in answered)}/{len(answered)} cited")

    w = 78
    print("=" * w); print("HEALTH CHECK — RETRIEVAL + GENERATION"); print("=" * w)
    for lab, st, det in HEALTH:
        print(f"  [{st}] {lab:<44} {det}")
    print("-" * w)
    nf = sum(1 for _, s, _ in HEALTH if s == "FAIL")
    print(f"  {len(HEALTH)-nf} pass / {nf} fail")
    if nf:
        print("\n  A G2 failure is the serious one: the model invented a figure that")
        print("  is not in any retrieved chunk. That is the §10 hallucination rule")
        print("  breaking, and it must be reported in M4, not tuned away quietly.")
else:
    print("[SKIP] RUN_HEALTH = False")

      unsupported ['150'] in: how much urea should be applied in wheat at tillering stage
      unsupported ['50'] in: टमाटर के पौधे में पत्तियां मुड़ रही हैं क्या करें
      unsupported ['150'] in: how much urea should be applied in wheat at tillering stage
HEALTH CHECK — RETRIEVAL + GENERATION
  [PASS] R1 filter kcc+crop=rice (canon 'Paddy (Dhan)') 5 hits
  [PASS] R2 district canon allahabad->prayagraj       5 hits
  [PASS] R3 domain separation margin > 0.02           in-min=0.593 off-max=0.522 margin=+0.070
  [PASS] G1 abstain never invokes the generator       0/4 generated
  [FAIL] G2 every number traces to context            15/84 answers contain an unsupported number
  [PASS] G3 Hindi question -> Hindi answer            0/21 mismatched
  [PASS] G4 fallback tier carries the KVK line        35 fallback-tier answers
  [PASS] G5 answered queries cite their sources       73/84 cited
------------------------------------------------------------------------------
  7 pass / 1 fail

  A G

## 15. Latency Decomposition

M3 §12.3 calls the retrieval-vs-budget conflict *"the most important open item in
this report"* — measured p50 466 ms against a 200–300 ms end-to-end target. That
figure was never broken down.

This splits it into **embed / search / generate**, which decides what to fix.
bge-m3 is 568M parameters; if query embedding dominates, the answer is ONNX/FP16
plus a Redis embedding cache, not a retrieval redesign.

Expect generation to be seconds, not milliseconds, on this hardware. That is not
a failure to hide — Appendix A specifies FP8 on vLLM, and this is 4-bit
bitsandbytes on a T4. Report it the way §12.3 reports retrieval: measured on
Colab, target hardware unmeasured, and the gap is itself the finding.

In [ ]:
LATENCY = {}
_lat_rows = GEN_SWEEP_ROWS or EXAMPLE_ROWS
_lat_rows = [r for r in _lat_rows if r.get("timing")]

if _lat_rows:
    def pct(key, p):
        vals = [r["timing"].get(key, 0) for r in _lat_rows if r["timing"].get(key) is not None]
        return float(np.percentile(vals, p)) if vals else 0.0

    LATENCY = {
        "n": len(_lat_rows),
        "embed_ms":  {"p50": round(pct("embed_ms", 50), 1),  "p95": round(pct("embed_ms", 95), 1)},
        "search_ms": {"p50": round(pct("search_ms", 50), 1), "p95": round(pct("search_ms", 95), 1)},
        "gen_ms":    {"p50": round(pct("gen_ms", 50), 1),    "p95": round(pct("gen_ms", 95), 1)},
        "total_ms":  {"p50": round(pct("total_ms", 50), 1),  "p95": round(pct("total_ms", 95), 1)},
    }
    w = 62
    print("=" * w); print(f"LATENCY DECOMPOSITION  (n={LATENCY['n']})"); print("=" * w)
    print(f"{'stage':<14}{'p50 ms':>12}{'p95 ms':>12}{'share of p50':>16}")
    print("-" * w)
    tot = max(LATENCY["total_ms"]["p50"], 1e-9)
    for stage in ("embed_ms", "search_ms", "gen_ms"):
        v = LATENCY[stage]
        print(f"{stage:<14}{v['p50']:>12.1f}{v['p95']:>12.1f}{100*v['p50']/tot:>15.1f}%")
    print("-" * w)
    print(f"{'total':<14}{LATENCY['total_ms']['p50']:>12.1f}{LATENCY['total_ms']['p95']:>12.1f}")
    print("\nM3 §1.2 target: 200-300 ms end to end. Compare honestly and record the")
    print("hardware — Appendix A assumes FP8 + vLLM; this is 4-bit on a Colab GPU.")
else:
    print("[SKIP] no timed rows — run §8 or §12 first")

LATENCY DECOMPOSITION  (n=84)
stage               p50 ms      p95 ms    share of p50
--------------------------------------------------------------
embed_ms              21.0        31.3            0.2%
search_ms             19.2        31.2            0.2%
gen_ms             10503.1     14836.2           99.6%
--------------------------------------------------------------
total              10540.5     14876.8

M3 §1.2 target: 200-300 ms end to end. Compare honestly and record the
hardware — Appendix A assumes FP8 + vLLM; this is 4-bit on a Colab GPU.


## 16. Export Artifacts for Milestone 5

M5 is evaluation and error analysis. Everything it needs is dumped here so it is
analysis over a file, with no reruns and no GPU:

| File | Contents |
|---|---|
| `generations.jsonl` | one row per generation: query, tier, retrieved ids, scores, answer, timing, config |
| `retrieval_sweep.json` | §11 table |
| `generation_sweep.json` | §12 + §13 tables |
| `health_check.json` | §14 results |
| `latency.json` | §15 decomposition |
| `run_manifest.json` | model ids, quantization, thresholds, fusion weights, git-less config hash |

The run manifest exists for the same reason `manifest.json` does (§9.11): a set of
results whose configuration has to be guessed at is not reproducible, and every
wrong guess about prefixes, thresholds or quantization fails silently.

In [ ]:
import hashlib

if EXPORT_ARTIFACTS:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_rows = []
    for r in (GEN_SWEEP_ROWS or []) + (EXAMPLE_ROWS or []):
        r = dict(r)
        r.pop("_ctx_blob", None)
        r.pop("_ctx_texts", None)
        all_rows.append(r)

    jsonl_path = os.path.join(OUTPUT_DIR, "generations.jsonl")
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for r in all_rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

    run_manifest = {
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "notebook": "10_rag_generation.ipynb",
        "retrieval": {
            "embed_model": EMBED_MODEL_NAME, "embed_dim": EMBED_DIM,
            "query_prefix": QUERY_PREFIX, "doc_prefix": DOC_PREFIX,
            "max_seq_length": MODEL_MAX_TOKENS, "collection": COLLECTION_NAME,
            "n_points": int(info.points_count),
            "tiers": {"grounded": TIER_GROUNDED, "fallback": TIER_FALLBACK},
            "fusion_weights": FUSION_WEIGHTS, "top_k_default": TOP_K_DEFAULT,
        },
        "generation": {**GEN_LOAD_INFO, "baseline": GEN_BASELINE,
                       "available": bool(GEN_AVAILABLE)},
        "eval_set_size": len(EVAL_SET),
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    run_manifest["config_hash"] = hashlib.sha256(
        json.dumps(run_manifest, sort_keys=True, default=str).encode()).hexdigest()[:12]

    for name, obj in [
        ("run_manifest.json",     run_manifest),
        ("retrieval_sweep.json",  RETRIEVAL_SWEEP),
        ("generation_sweep.json", {"ofat": GEN_SWEEP, "quantization": QUANT_SWEEP}),
        ("health_check.json",     [{"check": a, "status": b, "detail": c} for a, b, c in HEALTH]),
        ("latency.json",          LATENCY),
    ]:
        with open(os.path.join(OUTPUT_DIR, name), "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

    print(f"exported to {OUTPUT_DIR}")
    print(f"   generations.jsonl      {len(all_rows)} rows")
    for name in ("run_manifest.json", "retrieval_sweep.json", "generation_sweep.json",
                 "health_check.json", "latency.json"):
        p = os.path.join(OUTPUT_DIR, name)
        print(f"   {name:<24} {os.path.getsize(p):,} bytes")
    print(f"\nconfig_hash: {run_manifest['config_hash']}  "
          f"(quote this in the M4 report next to every number above)")
else:
    print("[SKIP] EXPORT_ARTIFACTS = False")

exported to /content/drive/MyDrive/rag_generation_m4
   generations.jsonl      90 rows
   run_manifest.json        1,049 bytes
   retrieval_sweep.json     1,070 bytes
   generation_sweep.json    1,944 bytes
   health_check.json        977 bytes
   latency.json             240 bytes

config_hash: bbbb78ca111b  (quote this in the M4 report next to every number above)


## 17. Ask Anything

Edit and re-run this one cell. Filters are the same as `09`: `intent`,
`source_type`, `crop`, `district`, `season`, `query_type`, `doc_category`,
`language`, `year_from`, `only_tables`, plus the generation knobs
`ctx_top_k`, `temperature`, `max_new_tokens`.

In [ ]:
MY_QUERIES  = [{"q":"sarkar ki sbse nayi scheme koinsi hai","intent":"policy"}
,{"q":"how do I repair my motorcycle engine","intent":"general"},
{"q":"hi how are you","intent":"general"},{"q":"i need to know latest farming practices and which crops could i benefit from if am in UP and also i want to know about government policies to beneift from","intent":"general"}
               ]# a mix of of policy and field practice query
TIER_GROUNDED = 0.66
TIER_FALLBACK = 0.56
for query in MY_QUERIES:
  MY_PARAMS = dict(intent=query["intent"], ctx_top_k=5, temperature=0.3, max_new_tokens=200)
  _row = answer_scored(query["q"], **MY_PARAMS)
  show(_row)
  print("scores:", json.dumps(_row["scores"], ensure_ascii=False))




# Raw JSON — exactly the row written to generations.jsonl
# print(json.dumps({k: v for k, v in _row.items() if not k.startswith('_')},
#                  indent=2, ensure_ascii=False, default=str))

Q     : sarkar ki sbse nayi scheme koinsi hai
tier  : abstain_out_of_scope   top=0.502   lang=en   generated=False
timing: embed=30ms  search=1217ms  gen=0ms  total=0ms

ANSWER:
This question is outside the agricultural knowledge base I can answer from. Please ask about crops, pests, fertilisers, or government agriculture schemes in Uttar Pradesh.

scores: {"numeric_grounding": 1.0, "unsupported_numbers": [], "language_match": 1.0, "has_citation": false, "out_tokens": null, "length_ok": null}
Q     : how do I repair my motorcycle engine
tier  : abstain_out_of_scope   top=0.522   lang=en   generated=False
timing: embed=29ms  search=26ms  gen=0ms  total=0ms

ANSWER:
This question is outside the agricultural knowledge base I can answer from. Please ask about crops, pests, fertilisers, or government agriculture schemes in Uttar Pradesh.

scores: {"numeric_grounding": 1.0, "unsupported_numbers": [], "language_match": 1.0, "has_citation": false, "out_tokens": null, "length_ok": null}
Q     :

In [ ]:
# --- FORCE OVERRIDE: Lower thresholds to test this specific query ---
# Defaults were: grounded=0.66, fallback=0.56
# Set them low enough so your query crosses the generation bar.
TIER_GROUNDED = 0.66 # If score >= 0.66, you get a "grounded" answer
TIER_FALLBACK = 0.45   # If score >= 0.45, you get a "fallback" answer (with disclaimer)
# -------------------------------------------------------------------
MY_QUERIES  = [{"q":"sarkar ki sbse nayi scheme koinsi hai","intent":"policy"}
,{"q":"how do I repair my motorcycle engine","intent":"general"},
{"q":"hi how are you","intent":"general"},{"q":"i need to know latest farming practices and which crops could i benefit from if am in UP and also i want to know about government policies to beneift from","intent":"field_practice"}]

# IMPORTANT: For scheme-related queries, use "policy" intent so PDFs get higher weight!
for query in MY_QUERIES:
  MY_PARAMS = dict(intent=query["intent"], ctx_top_k=10, temperature=0.3, max_new_tokens=200)
  _row = answer_scored(query["q"], **MY_PARAMS)
  show(_row)
  print("scores:", json.dumps(_row["scores"], ensure_ascii=False))

---
## What to Take Into the M4 Report

**Sections this notebook feeds:**

| M4 requirement | Where |
|---|---|
| Training configuration | §0 config + §16 run manifest (generation is zero-shot — state that plainly) |
| Hyperparameter experiments | §11 retrieval table, §12 generation table, §13 quantization |
| Generalization / stability | temperature and `ctx_top_k` axes; the tier gate as a stability mechanism |
| Quantitative results | §11–13 tables, §15 latency |
| Qualitative results + sample outputs | §8 worked examples, `generations.jsonl` |
| Artifacts | §16 file list + `config_hash` |
| Key findings | §14 health check, especially any G2 failure |

**Three things to write down honestly, whatever the numbers say:**

1. **This generator is not trained.** It is the §5.4 off-the-shelf fallback, run
   zero-shot. The §7.5 distillation set does not exist, so distillation is deferred
   with reasons rather than reported as done. The value of this run is that it is
   the **control** the M5 fine-tuned student gets measured against.

2. **Any G2 failure is a result, not a bug to hide.** An invented dosage in a
   fluent, cited answer is exactly the class of silent failure M3 §16.2 warns about
   — clean run, plausible output, no error. Report the count.

3. **Latency will miss the §1.2 budget, and the reason is the hardware.** Appendix
   A specifies FP8 on vLLM; this is 4-bit bitsandbytes on a Colab GPU. Report the
   §15 split and let the decomposition say which component to fix, rather than
   concluding the architecture is wrong from a measurement it was never scoped to.

**Next step after this notebook:** build the §7.5 distillation pair set from
`generations.jsonl` plus real KCC answers, then QLoRA the same checkpoint and
re-run every sweep here unchanged. Same eval set, same scorers, same config hash
discipline — that is what makes the before/after comparison mean something.